# **05_modelos_ml**

Este notebook implementa modelos de Machine Learning para la predicción de la demanda
eléctrica diaria, utilizando un enfoque tabular con variables temporales, meteorológicas
y de calendario.

## Objetivos
- Construir un dataset de modelado para Machine Learning.
- Generar variables derivadas de la serie temporal.
- Entrenar y evaluar un modelo Random Forest.
- Entrenar y evaluar un modelo XGBoost.
- Comparar el desempeño de ambos modelos en el conjunto de prueba.
- Analizar la importancia de variables.
- Generar tablas de hiperparámetros y resultados útiles para el TFM.

**BLOQUE 2 — Librerías y configuración**

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

from xgboost import XGBRegressor

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)
pd.set_option("display.float_format", lambda x: f"{x:,.6f}")

**BLOQUE 3 — Rutas**

In [ ]:
# ==========================================
# BLOQUE 3 — Configuración de rutas
# ==========================================

from pathlib import Path

# Detectar si está dentro de notebooks/
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
OUTPUTS_FIGURES = PROJECT_ROOT / "outputs" / "figuras"
OUTPUTS_TABLES = PROJECT_ROOT / "outputs" / "tablas"

# Crear carpetas si no existen
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
OUTPUTS_FIGURES.mkdir(parents=True, exist_ok=True)
OUTPUTS_TABLES.mkdir(parents=True, exist_ok=True)

# Archivo de entrada
INPUT_FILE = DATA_PROCESSED / "data_daily_ready.csv"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Archivo de entrada:", INPUT_FILE)

# Validación
if not INPUT_FILE.exists():
    raise FileNotFoundError(
        f"No se encontró el archivo: {INPUT_FILE}\n"
        "Verifica que el archivo exista en data/processed/"
    )

**BLOQUE 4 — Carga de datos**

In [ ]:
df = pd.read_csv(INPUT_FILE, parse_dates=["date"])

DATE_COL = "date"
TARGET_COL = "target_kwh"

df = df.sort_values(DATE_COL).set_index(DATE_COL)

print(" Nulos en dataset original df:")
display(df.isna().sum().sort_values(ascending=False).head(30))

print("Dimensión del dataset:", df.shape)
print("Frecuencia inferida:", pd.infer_freq(df.index))

display(df.head())

**BLOQUE 5 — División temporal train/test**

In [ ]:
train = df.loc["2022-01-01":"2024-12-31"].copy()
test = df.loc["2025-01-01":"2025-12-10"].copy()

print("Train shape:", train.shape)
print("Test shape :", test.shape)
print("Rango train:", train.index.min(), "->", train.index.max())
print("Rango test :", test.index.min(), "->", test.index.max())

**BLOQUE 6 — Funciones auxiliares**

In [ ]:
def mape(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    mask = y_true != 0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

def compute_metrics(y_true, y_pred, model_name):
    return {
        "model": model_name,
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "MAPE": mape(y_true, y_pred)
    }

**BLOQUE 7 — Feature engineering**

In [ ]:
ml_df = df.copy()

# Calendario
ml_df["year"] = ml_df.index.year
ml_df["month"] = ml_df.index.month
ml_df["weekofyear"] = ml_df.index.isocalendar().week.astype(int)
ml_df["dow"] = ml_df.index.dayofweek

# Variables cíclicas
ml_df["dow_sin"] = np.sin(2 * np.pi * ml_df["dow"] / 7)
ml_df["dow_cos"] = np.cos(2 * np.pi * ml_df["dow"] / 7)
ml_df["month_sin"] = np.sin(2 * np.pi * ml_df["month"] / 12)
ml_df["month_cos"] = np.cos(2 * np.pi * ml_df["month"] / 12)

# Lags (REDUCIDOS)
for lag in [1, 2, 7, 14]:
    ml_df[f"y_lag_{lag}"] = ml_df[TARGET_COL].shift(lag)

# Rolling (REDUCIDO)
for window in [7, 14]:
    ml_df[f"y_roll_mean_{window}"] = ml_df[TARGET_COL].shift(1).rolling(window=window).mean()
    ml_df[f"y_roll_std_{window}"] = ml_df[TARGET_COL].shift(1).rolling(window=window).std()

# Diferencias
ml_df["y_diff_1"] = ml_df[TARGET_COL].diff(1)
ml_df["y_diff_7"] = ml_df[TARGET_COL].diff(7)

print("Dimensión con features:", ml_df.shape)

**BLOQUE 8 — Selección de variables**

In [ ]:
feature_cols = [
    # meteorológicas
    "t_mean_c", "t_max_c", "t_min_c", "rh_mean_pct", "precip_mm", "sun_hours", "sst_c",

    # calendario
    "year", "month", "weekofyear", "dow",
    "dow_sin", "dow_cos", "month_sin", "month_cos",
    "is_weekend", "is_business_day", "is_holiday", "is_pre_holiday", "is_post_holiday", "is_long_weekend",
    "holiday_name",

    # lags
    "y_lag_1", "y_lag_2", "y_lag_7", "y_lag_14",

    # rolling
    "y_roll_mean_7", "y_roll_mean_14",
    "y_roll_std_7", "y_roll_std_14",

    # diferencias
    "y_diff_1", "y_diff_7"
]

missing_features = [col for col in feature_cols if col not in ml_df.columns]
if missing_features:
    raise ValueError(f"Faltan features: {missing_features}")

# Corregir holiday_name
ml_df["holiday_name"] = ml_df["holiday_name"].fillna("NoHoliday")

# Construcción dataset ML
ml_model_df = ml_df[[TARGET_COL] + feature_cols].copy()

print("Filas antes de limpieza:", len(ml_model_df))

# Eliminar solo filas sin target
ml_model_df = ml_model_df.dropna(subset=[TARGET_COL])

# Eliminar filas iniciales afectadas por lags/rolling
ml_model_df = ml_model_df.iloc[14:].copy()

print("Filas después de limpieza:", len(ml_model_df))
print("Dimensión final para ML:", ml_model_df.shape)

display(ml_model_df.head())

**BLOQUE 9 — Rehacer split sobre dataset de ML**

In [ ]:
train_ml = ml_model_df.loc["2022-01-01":"2024-12-31"].copy()
test_ml = ml_model_df.loc["2025-01-01":"2025-12-10"].copy()

X_train_ml = train_ml[feature_cols].copy()
y_train_ml = train_ml[TARGET_COL].copy()

X_test_ml = test_ml[feature_cols].copy()
y_test_ml = test_ml[TARGET_COL].copy()

print("X_train_ml:", X_train_ml.shape)
print("X_test_ml :", X_test_ml.shape)
print("y_train_ml:", y_train_ml.shape)
print("y_test_ml :", y_test_ml.shape)

**BLOQUE 10 — Tipos de variables**

In [ ]:
categorical_features = ["holiday_name"]
numeric_features = [col for col in feature_cols if col not in categorical_features]

print("Nº features numéricas:", len(numeric_features))
print("Nº features categóricas:", len(categorical_features))

**BLOQUE 11 — Preprocesamiento común**

In [ ]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

**BLOQUE 12 — Validación temporal para tuning**

In [ ]:
# ==========================================
# BLOQUE 12 — Validación temporal para tuning
# ==========================================

from sklearn.model_selection import TimeSeriesSplit, GridSearchCV

# Validación temporal interna solo sobre train
tscv = TimeSeriesSplit(n_splits=4)

print("Configuración TimeSeriesSplit:")
for i, (tr_idx, val_idx) in enumerate(tscv.split(X_train_ml), start=1):
    print(f"Fold {i}: train={len(tr_idx)} | valid={len(val_idx)}")

**BLOQUE 13 — GridSearchCV Random Forest**

In [ ]:
# ==========================================
# BLOQUE 13 — GridSearchCV Random Forest
# ==========================================

from sklearn.ensemble import RandomForestRegressor

rf_pipe = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(random_state=42, n_jobs=-1))
])

rf_param_grid = {
    "model__n_estimators": [300, 600, 900],
    "model__max_depth": [None, 10, 20],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 4],
    "model__max_features": [1.0, "sqrt"]
}

rf_grid = GridSearchCV(
    estimator=rf_pipe,
    param_grid=rf_param_grid,
    cv=tscv,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1,
    verbose=1
)

rf_grid.fit(X_train_ml, y_train_ml)

print("Mejores parámetros RF:")
print(rf_grid.best_params_)
print("Mejor score CV RF (RMSE negativo):", rf_grid.best_score_)

rf_cv_results = pd.DataFrame(rf_grid.cv_results_)
rf_cv_results = rf_cv_results.sort_values("rank_test_score").reset_index(drop=True)

display(
    rf_cv_results[
        [
            "rank_test_score",
            "mean_test_score",
            "std_test_score",
            "param_model__n_estimators",
            "param_model__max_depth",
            "param_model__min_samples_split",
            "param_model__min_samples_leaf",
            "param_model__max_features"
        ]
    ].head(10)
)

rf_cv_results.to_csv(OUTPUTS_TABLES / "table_rf_gridsearch_results.csv", index=False)

**BLOQUE 14 — Entrenamiento final Random Forest**

In [ ]:
# ==========================================
# BLOQUE 14 — Entrenamiento final Random Forest
# ==========================================

rf_model = rf_grid.best_estimator_

rf_pred = rf_model.predict(X_test_ml)

rf_metrics = compute_metrics(y_test_ml, rf_pred, "RandomForest")
display(pd.DataFrame([rf_metrics]))

# Guardar mejores hiperparámetros
rf_best_params = pd.DataFrame([rf_grid.best_params_])
rf_best_params.to_csv(OUTPUTS_TABLES / "table_rf_best_params.csv", index=False)

**BLOQUE 15 — Gráfico RF**

---



In [ ]:
# ==========================================
# BLOQUE 15 — Gráfico Random Forest
# ==========================================

fig, ax = plt.subplots(figsize=(15, 4.8))

ax.plot(y_test_ml.index, y_test_ml.values, label="Real", linewidth=1.2)
ax.plot(y_test_ml.index, rf_pred, label="Predicción Random Forest", linewidth=1.2)

ax.set_title("Predicción en test - Random Forest", fontsize=13)
ax.set_xlabel("Fecha")
ax.set_ylabel("Demanda eléctrica diaria (kWh)")
ax.grid(True, alpha=0.3)
ax.legend(frameon=False)

ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))

plt.tight_layout()
plt.savefig(OUTPUTS_FIGURES / "forecast_test_random_forest_tuned.png", dpi=300, bbox_inches="tight")
plt.show()

**BLOQUE 16 — GridSearchCV XGBoost**

In [ ]:
# ==========================================
# BLOQUE 16 — GridSearchCV XGBoost
# ==========================================

from xgboost import XGBRegressor

xgb_pipe = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", XGBRegressor(
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1
    ))
])

xgb_param_grid = {
    "model__n_estimators": [300, 500, 700],
    "model__learning_rate": [0.03, 0.05, 0.1],
    "model__max_depth": [3, 5, 7],
    "model__subsample": [0.8, 1.0],
    "model__colsample_bytree": [0.8, 1.0],
    "model__min_child_weight": [1, 3, 5]
}

xgb_grid = GridSearchCV(
    estimator=xgb_pipe,
    param_grid=xgb_param_grid,
    cv=tscv,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1,
    verbose=1
)

xgb_grid.fit(X_train_ml, y_train_ml)

print("Mejores parámetros XGBoost:")
print(xgb_grid.best_params_)
print("Mejor score CV XGBoost (RMSE negativo):", xgb_grid.best_score_)

xgb_cv_results = pd.DataFrame(xgb_grid.cv_results_)
xgb_cv_results = xgb_cv_results.sort_values("rank_test_score").reset_index(drop=True)

display(
    xgb_cv_results[
        [
            "rank_test_score",
            "mean_test_score",
            "std_test_score",
            "param_model__n_estimators",
            "param_model__learning_rate",
            "param_model__max_depth",
            "param_model__subsample",
            "param_model__colsample_bytree",
            "param_model__min_child_weight"
        ]
    ].head(10)
)

xgb_cv_results.to_csv(OUTPUTS_TABLES / "table_xgb_gridsearch_results.csv", index=False)

**BLOQUE 17 — Entrenamiento final XGBoost**

In [ ]:
# ==========================================
# BLOQUE 17 — Entrenamiento final XGBoost
# ==========================================

xgb_model = xgb_grid.best_estimator_

xgb_pred = xgb_model.predict(X_test_ml)

xgb_metrics = compute_metrics(y_test_ml, xgb_pred, "XGBoost")
display(pd.DataFrame([xgb_metrics]))

# Guardar mejores hiperparámetros
xgb_best_params = pd.DataFrame([xgb_grid.best_params_])
xgb_best_params.to_csv(OUTPUTS_TABLES / "table_xgb_best_params.csv", index=False)

**BLOQUE 18 — Gráfico XGBoost**

In [ ]:
# ==========================================
# BLOQUE 18 — Gráfico XGBoost
# ==========================================

fig, ax = plt.subplots(figsize=(15, 4.8))

ax.plot(y_test_ml.index, y_test_ml.values, label="Real", linewidth=1.2)
ax.plot(y_test_ml.index, xgb_pred, label="Predicción XGBoost", linewidth=1.2)

ax.set_title("Predicción en test - XGBoost", fontsize=13)
ax.set_xlabel("Fecha")
ax.set_ylabel("Demanda eléctrica diaria (kWh)")
ax.grid(True, alpha=0.3)
ax.legend(frameon=False)

ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))

plt.tight_layout()
plt.savefig(OUTPUTS_FIGURES / "forecast_test_xgboost_tuned.png", dpi=300, bbox_inches="tight")
plt.show()

**BLOQUE 19 — Comparación RF vs XGB**

In [ ]:
# ==========================================
# BLOQUE 19 — Comparación RF vs XGB
# ==========================================

ml_comparison = pd.DataFrame([rf_metrics, xgb_metrics]).sort_values("RMSE").reset_index(drop=True)
display(ml_comparison)

ml_comparison.to_csv(OUTPUTS_TABLES / "table_ml_models_comparison_tuned.csv", index=False)

**BLOQUE 20 — Tabla de hiperparámetros**

In [ ]:
# ==========================================
# BLOQUE 20 — Predicciones exportables
# ==========================================

predictions_ml = pd.DataFrame({
    "date": y_test_ml.index,
    "y_true": y_test_ml.values,
    "y_pred_rf": rf_pred,
    "y_pred_xgb": xgb_pred
})

display(predictions_ml.head())
predictions_ml.to_csv(OUTPUTS_TABLES / "table_ml_test_predictions_tuned.csv", index=False)

**BLOQUE 21 — Importancia de variables**

In [ ]:
# ==========================================
# BLOQUE 21 — Importancia de variables
# ==========================================

feature_names = rf_model.named_steps["preprocessor"].get_feature_names_out()

rf_importances = rf_model.named_steps["model"].feature_importances_
xgb_importances = xgb_model.named_steps["model"].feature_importances_

rf_importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance_rf": rf_importances
}).sort_values("importance_rf", ascending=False).reset_index(drop=True)

xgb_importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance_xgb": xgb_importances
}).sort_values("importance_xgb", ascending=False).reset_index(drop=True)

display(rf_importance_df.head(15))
display(xgb_importance_df.head(15))

rf_importance_df.to_csv(OUTPUTS_TABLES / "table_feature_importance_rf_tuned.csv", index=False)
xgb_importance_df.to_csv(OUTPUTS_TABLES / "table_feature_importance_xgb_tuned.csv", index=False)